## Классификация: превышает ли значение SI медианное значение выборки

In [10]:
import pandas as pd

import sys
sys.path.append('../src')
from preprocessing import DatasetPreprocessor
from metrics_calculator import MetricsCalculator

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [4]:
df = pd.read_excel('../data/raw/classicMLCourseWorkData.xlsx')

# удаляем строки с пропусками
clean_df = df.dropna().copy()

# медиана SI
si_median = clean_df['SI'].median()

# бинарный таргет
y = (clean_df['SI'] > si_median).astype(int)

# признаки
X = clean_df.drop(columns=['SI', 'IC50, mM', 'CC50, mM'])

# Класс расчета метрик
metrics_helper = MetricsCalculator()

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#### Логистическая регрессия

In [6]:
logreg_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column=None,
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        LogisticRegression(max_iter=1000)
    )
])

# обучение
logreg_pipeline.fit(X_train, y_train)

# предсказания
logreg_y_train_pred = logreg_pipeline.predict(X_train)
logreg_y_test_pred = logreg_pipeline.predict(X_test)

# метрики
logreg_train_metrics = metrics_helper.classification_metrics(y_train, logreg_y_train_pred)
logreg_test_metrics = metrics_helper.classification_metrics(y_test, logreg_y_test_pred)

print('Метрики на train:')
display(logreg_train_metrics)

print('Метрики на test:')
display(logreg_test_metrics)

Метрики на train:


{'accuracy': 0.7355889724310777,
 'precision': 0.7338308457711443,
 'recall': 0.7393483709273183,
 'f1': 0.7365792759051186}

Метрики на test:


{'accuracy': 0.625,
 'precision': 0.6288659793814433,
 'recall': 0.61,
 'f1': 0.6192893401015228}

- Модель показывает среднее качество на train (accuracy $\approx 0.74$) и значительно хуже на test (accuracy $\approx 0.63$)

**Вывод:** логистическая регрессия показывает слабое качество, требуется более сложная модель

#### Случайный лес

In [7]:
rf_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column=None,
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
])

# обучение
rf_pipeline.fit(X_train, y_train)

# предсказания
rf_y_train_pred = rf_pipeline.predict(X_train)
rf_y_test_pred = rf_pipeline.predict(X_test)

# метрики
rf_train_metrics = metrics_helper.classification_metrics(y_train, rf_y_train_pred)
rf_test_metrics = metrics_helper.classification_metrics(y_test, rf_y_test_pred)

print('Метрики на train:')
display(rf_train_metrics)

print('Метрики на test:')
display(rf_test_metrics)

Метрики на train:


{'accuracy': 0.9548872180451128,
 'precision': 0.9373493975903614,
 'recall': 0.974937343358396,
 'f1': 0.9557739557739557}

Метрики на test:


{'accuracy': 0.63,
 'precision': 0.6413043478260869,
 'recall': 0.59,
 'f1': 0.6145833333333334}

- Модель показывает высокое качество на train (accuracy $\approx 0.95$), но низкое качество на test (accuracy $\approx 0.63$)
- Наблюдается сильное переобучение

**Вывод:** RandomForest в текущем виде не даёт улучшения, требуется подбор гиперпараметров

In [8]:
# сетка гиперпараметров
rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 0.5]
}

# GridSearch
rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# обучение
rf_grid_search.fit(X_train, y_train)

# лучшая модель
best_rf_pipeline = rf_grid_search.best_estimator_

print('Лучшие параметры:')
print(rf_grid_search.best_params_)

# предсказания
best_rf_y_train_pred = best_rf_pipeline.predict(X_train)
best_rf_y_test_pred = best_rf_pipeline.predict(X_test)

# метрики
best_rf_train_metrics = metrics_helper.classification_metrics(y_train, best_rf_y_train_pred)
best_rf_test_metrics = metrics_helper.classification_metrics(y_test, best_rf_y_test_pred)

print('Метрики на train:')
display(best_rf_train_metrics)

print('Метрики на test:')
display(best_rf_test_metrics)

Лучшие параметры:
{'model__max_depth': 10, 'model__max_features': 0.5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Метрики на train:


{'accuracy': 0.9511278195488722,
 'precision': 0.9326923076923077,
 'recall': 0.9724310776942355,
 'f1': 0.9521472392638037}

Метрики на test:


{'accuracy': 0.63,
 'precision': 0.6354166666666666,
 'recall': 0.61,
 'f1': 0.6224489795918368}

- Подбор гиперпараметров не привёл к заметному улучшению качества на test (accuracy $\approx 0.63$, f1 $\approx 0.62$)
- Качество на train остаётся высоким, разрыв между train и test сохраняется

**Вывод:** настройка RandomForest не решает проблему качества

In [9]:
# вероятности положительного класса
best_rf_y_test_proba = best_rf_pipeline.predict_proba(X_test)[:, 1]

# полный набор метрик
best_rf_full_test_metrics = metrics_helper.full_classification_metrics(
    y_true=y_test,
    y_pred=best_rf_y_test_pred,
    y_proba=best_rf_y_test_proba
)

print('Полный набор метрик на test:')
display(best_rf_full_test_metrics)

Полный набор метрик на test:


{'accuracy': 0.63,
 'precision': 0.6354166666666666,
 'recall': 0.61,
 'f1': 0.6224489795918368,
 'roc_auc': 0.67825,
 'tn': np.int64(65),
 'fp': np.int64(35),
 'fn': np.int64(39),
 'tp': np.int64(61)}

- Модель показывает низкое качество на test (accuracy $\approx 0.63$, f1 $\approx 0.62$)
- ROC-AUC $\approx 0.68$, слабое разделение классов
- Количество ошибок значительное: FP = 35, FN = 39

**Вывод:** RandomForest не даёт приемлемого качества для задачи классификации SI

In [11]:
gb_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column=None,
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        )
    )
])

# обучение
gb_pipeline.fit(X_train, y_train)

# предсказания
gb_y_train_pred = gb_pipeline.predict(X_train)
gb_y_test_pred = gb_pipeline.predict(X_test)

# метрики
gb_train_metrics = metrics_helper.classification_metrics(y_train, gb_y_train_pred)
gb_test_metrics = metrics_helper.classification_metrics(y_test, gb_y_test_pred)

print('Метрики на train:')
display(gb_train_metrics)

print('Метрики на test:')
display(gb_test_metrics)

Метрики на train:


{'accuracy': 0.9022556390977443,
 'precision': 0.8982630272952854,
 'recall': 0.9072681704260651,
 'f1': 0.9027431421446384}

Метрики на test:


{'accuracy': 0.665,
 'precision': 0.6813186813186813,
 'recall': 0.62,
 'f1': 0.6492146596858639}

- Модель показывает высокое качество на train (accuracy $\approx 0.90$), но заметно хуже на test (accuracy $\approx 0.67$)
- Наблюдается переобучение
- Качество немного выше, чем у RandomForest ($\approx 0.63$), но остаётся низким

**Вывод:** GradientBoosting не даёт существенного улучшения качества

In [12]:
# вероятности
gb_y_test_proba = gb_pipeline.predict_proba(X_test)[:, 1]

# полный набор метрик
gb_full_test_metrics = metrics_helper.full_classification_metrics(
    y_true=y_test,
    y_pred=gb_y_test_pred,
    y_proba=gb_y_test_proba
)

print('Полный набор метрик на test:')
display(gb_full_test_metrics)

Полный набор метрик на test:


{'accuracy': 0.665,
 'precision': 0.6813186813186813,
 'recall': 0.62,
 'f1': 0.6492146596858639,
 'roc_auc': 0.66605,
 'tn': np.int64(71),
 'fp': np.int64(29),
 'fn': np.int64(38),
 'tp': np.int64(62)}

- Все модели показывают низкое качество
- ROC-AUC находится на уровне ~0.66–0.68, модели слабо разделяют классы
- Более сложные модели (RandomForest, GradientBoosting) не дают существенного улучшения по сравнению с логистической регрессией

**Вывод:** на поставленный вопрос о превышении SI медианного значения
невозможно дать надёжный ответ с использованием текущих признаков